# 197. MCP Tool Schema：版本协商、兼容迁移与契约测试怎样实现？

> **面试问题：工具输入/输出 schema 演进时，怎样协商协议版本、显式迁移旧参数、拒绝不兼容调用，并用 golden tests 防回归？**

## 先给结论

不要把 Agent 面试题答成框架 API：先定义状态、动作、权限、预算、版本和可判定的终态，再讨论 prompt、模型和并发扩展。下面用受控内存数据手写最小协议；小规模断言只证明实现合同，不代表线上模型效果、权限体系或安全等级。

## 一手资料

- [MCP Versioning](https://modelcontextprotocol.io/docs/learn/versioning)
- [MCP Tools](https://modelcontextprotocol.io/specification/draft/server/tools)
- [MCP Specification](https://modelcontextprotocol.io/specification/)

In [ ]:
notebook_contract = {"mode": "in-memory-demo", "oracle": "assertions", "production": "isolation-and-audit"}  # 执行本行的状态、计算或校验逻辑。
assert notebook_contract["mode"] == "in-memory-demo"  # 执行本行的状态、计算或校验逻辑。
assert notebook_contract["oracle"] == "assertions"  # 执行本行的状态、计算或校验逻辑。
assert "audit" in notebook_contract["production"]  # 执行本行的状态、计算或校验逻辑。
assert len(notebook_contract) == 3  # 执行本行的状态、计算或校验逻辑。


## 1. 问题拆解：工具 schema 是 Agent 的可执行 API 契约

模型能否产生看似正确的 JSON 不等于可安全调用工具。契约至少包含工具名、版本、必需输入、输出字段和语义约束；每一次调用都需在边界处校验，不能把 schema 描述仅留在 prompt 中。


In [ ]:
from dataclasses import dataclass  # 执行本行的状态、计算或校验逻辑。
@dataclass(frozen=True)  # 执行本行的状态、计算或校验逻辑。
class ToolSpec:  # 执行本行的状态、计算或校验逻辑。
    name: str  # 执行本行的状态、计算或校验逻辑。
    version: str  # 执行本行的状态、计算或校验逻辑。
    required_inputs: tuple  # 执行本行的状态、计算或校验逻辑。
    required_outputs: tuple  # 执行本行的状态、计算或校验逻辑。
weather_v1 = ToolSpec("weather.lookup", "2025-06-18", ("city",), ("temperature_c",))  # 执行本行的状态、计算或校验逻辑。
assert weather_v1.name == "weather.lookup"  # 执行本行的状态、计算或校验逻辑。
assert weather_v1.required_inputs == ("city",)  # 执行本行的状态、计算或校验逻辑。
assert weather_v1.version == "2025-06-18"  # 执行本行的状态、计算或校验逻辑。


## 2. 输入校验：拒绝缺字段、额外副作用字段与错误类型

这里实现极简结构校验：必需字段必须出现，未声明字段应在策略允许前拒绝。生产应使用正式 JSON Schema、枚举、数值边界、正则、跨字段条件和权限上下文校验。


In [ ]:
def validate_input(spec, arguments):  # 执行本行的状态、计算或校验逻辑。
    required = set(spec.required_inputs)  # 执行本行的状态、计算或校验逻辑。
    return required.issubset(arguments) and set(arguments).issubset(required) and all(isinstance(arguments[key], str) for key in required)  # 执行本行的状态、计算或校验逻辑。
assert validate_input(weather_v1, {"city": "Shanghai"})  # 执行本行的状态、计算或校验逻辑。
assert not validate_input(weather_v1, {})  # 执行本行的状态、计算或校验逻辑。
assert not validate_input(weather_v1, {"city": "Shanghai", "delete": "true"})  # 执行本行的状态、计算或校验逻辑。


## 3. 版本协商：会话只能选择双方支持的一个版本

MCP 使用基于日期的版本标识，并在初始化协商。客户端不能只读服务端最新版本就直接调用：应找出共同版本，记录在会话中，并在不存在交集时明确失败。


In [ ]:
def negotiate(client_versions, server_versions):  # 执行本行的状态、计算或校验逻辑。
    shared = sorted(set(client_versions) & set(server_versions), reverse=True)  # 执行本行的状态、计算或校验逻辑。
    if not shared:  # 执行本行的状态、计算或校验逻辑。
        raise ValueError("客户端与服务端没有共同协议版本")  # 执行本行的状态、计算或校验逻辑。
    return shared[0]  # 执行本行的状态、计算或校验逻辑。
agreed = negotiate(["2024-11-05", "2025-06-18"], ["2025-06-18"])  # 执行本行的状态、计算或校验逻辑。
assert agreed == "2025-06-18"  # 执行本行的状态、计算或校验逻辑。
assert negotiate(["2024-11-05"], ["2024-11-05"]) == "2024-11-05"  # 执行本行的状态、计算或校验逻辑。
assert len(agreed) == 10  # 执行本行的状态、计算或校验逻辑。


## 4. 演进：通过 adapter 做显式语义迁移

假设 v2 把 `city` 改为 `location`。兼容层应该是可测试的纯函数，而不是靠模型猜字段含义；如果新字段引入不同语义，例如经纬度或时区，也应要求调用方重新确认。


In [ ]:
weather_v2 = ToolSpec("weather.lookup", "2025-11-25", ("location",), ("temperature_c", "source"))  # 执行本行的状态、计算或校验逻辑。
def v1_to_v2(arguments):  # 执行本行的状态、计算或校验逻辑。
    if set(arguments) != {"city"}:  # 执行本行的状态、计算或校验逻辑。
        raise ValueError("只有完整 v1 输入可迁移")  # 执行本行的状态、计算或校验逻辑。
    return {"location": arguments["city"]}  # 执行本行的状态、计算或校验逻辑。
migrated = v1_to_v2({"city": "Shanghai"})  # 执行本行的状态、计算或校验逻辑。
assert migrated == {"location": "Shanghai"}  # 执行本行的状态、计算或校验逻辑。
assert validate_input(weather_v2, migrated)  # 执行本行的状态、计算或校验逻辑。
assert weather_v2.version > weather_v1.version  # 执行本行的状态、计算或校验逻辑。


## 5. 调用边界：输入、输出与工具错误要分别处理

工具返回错误不应伪装成有效答案；输出同样必须满足约定 schema。演示工具只返回固定结构，生产中还需处理超时、重试等级、速率限制、认证失败和不可信文本结果。


In [ ]:
def invoke(spec, arguments):  # 执行本行的状态、计算或校验逻辑。
    if not validate_input(spec, arguments):  # 执行本行的状态、计算或校验逻辑。
        return {"ok": False, "error": "input_schema"}  # 执行本行的状态、计算或校验逻辑。
    if spec.version == "2025-11-25":  # 执行本行的状态、计算或校验逻辑。
        return {"ok": True, "temperature_c": 28, "source": "demo-station"}  # 执行本行的状态、计算或校验逻辑。
    return {"ok": True, "temperature_c": 28}  # 执行本行的状态、计算或校验逻辑。
response = invoke(weather_v2, migrated)  # 执行本行的状态、计算或校验逻辑。
assert response["ok"] is True  # 执行本行的状态、计算或校验逻辑。
assert set(weather_v2.required_outputs).issubset(response)  # 执行本行的状态、计算或校验逻辑。
assert invoke(weather_v2, {"city": "Shanghai"})["ok"] is False  # 执行本行的状态、计算或校验逻辑。


## 6. 失败分支：不兼容版本和未知字段必须 fail closed

版本协商失败后降级为自然语言并继续调用旧工具，是常见事故来源。这里要求显式失败；调用者可以选择展示受限能力或请求升级，但不能猜测参数字段。


In [ ]:
try:  # 执行本行的状态、计算或校验逻辑。
    negotiate(["2025-11-25"], ["2024-11-05"])  # 执行本行的状态、计算或校验逻辑。
    assert False  # 执行本行的状态、计算或校验逻辑。
except ValueError:  # 执行本行的状态、计算或校验逻辑。
    assert True  # 执行本行的状态、计算或校验逻辑。
assert not validate_input(weather_v1, {"location": "Shanghai"})  # 执行本行的状态、计算或校验逻辑。
assert response["source"] == "demo-station"  # 执行本行的状态、计算或校验逻辑。


## 7. 契约测试：用 golden cases 防止升级悄悄破坏 Agent

每个 schema 版本都应有正例、负例和迁移样例。它们进入 CI 后，工具团队和 Agent 团队才能独立演进；还要测试真实客户端支持的 schema dialect 与错误码。


In [ ]:
golden_cases = [({"city": "Beijing"}, True), ({}, False), ({"city": 7}, False)]  # 执行本行的状态、计算或校验逻辑。
results = [validate_input(weather_v1, arguments) == expected for arguments, expected in golden_cases]  # 执行本行的状态、计算或校验逻辑。
assert all(results)  # 执行本行的状态、计算或校验逻辑。
assert len(results) == 3  # 执行本行的状态、计算或校验逻辑。
assert results[-1] is True  # 执行本行的状态、计算或校验逻辑。


## 8. 制品：把协议、schema、adapter 与调用 trace 一同版本化

一个可复放的工具调用至少要包含协商后的协议版本、工具 schema hash、adapter 版本和输入/输出摘要。这样才能定位问题属于模型选错工具、客户端没升级还是服务端破坏契约。


In [ ]:
import hashlib  # 执行本行的状态、计算或校验逻辑。
import json  # 执行本行的状态、计算或校验逻辑。
artifact = {"protocol": agreed, "tool": weather_v2.name, "schema": weather_v2.version, "adapter": "v1-to-v2"}  # 执行本行的状态、计算或校验逻辑。
schema_hash = hashlib.sha256(json.dumps(artifact, sort_keys=True).encode()).hexdigest()  # 执行本行的状态、计算或校验逻辑。
assert artifact["protocol"] == "2025-06-18"  # 执行本行的状态、计算或校验逻辑。
assert artifact["adapter"] == "v1-to-v2"  # 执行本行的状态、计算或校验逻辑。
assert len(schema_hash) == 64  # 执行本行的状态、计算或校验逻辑。


## 面试收束

回答时依次给出目标、状态合同、动作前校验、主路径、失败分支、指标、制品版本和生产替换点。可靠 Agent 不靠模型自述“完成”，而靠独立的状态 oracle、预算约束、审计和可复放 trace。
